# 04 · Partición de entrenamiento y validación

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Toma 90.000 comentarios del pool filtrado y los divide de forma aleatoria y estratificada por emoción (semilla 42) en 72.000 de entrenamiento y 18.000 de validación. También genera un conjunto de evaluación preliminar, que después fue reemplazado por la prueba final del notebook 06.

*Nombre durante el desarrollo: `04b_0_Preparar_Split_90k` (algunas salidas conservan esa numeración).*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

RUTA_DATA = '/content/drive/MyDrive/RUTA/A/data_prep'

EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
UMBRAL_CONFIANZA = 0.67

INCLUIR_GOLD_TEST_EN_VALIDACION = True

N_VALIDACION_FINAL_OBJETIVO = 10_000

Mounted at /content/drive


## 1. Cargar el pool filtrado y la muestra etiquetada a mano

In [3]:
df_pool = pd.read_csv(f'{RUTA_DATA}/pool_etiquetado.csv')
df_pool = df_pool[df_pool['label'] != 'no_valido'].copy()
df_pool = df_pool[df_pool['confianza'] >= UMBRAL_CONFIANZA].copy()
print(f"Pool filtrado (etiquetado por el LLM anotador, confianza >= {UMBRAL_CONFIANZA}): "
      f"{len(df_pool)} comentarios")

df_gold_dev = pd.read_csv(f'{RUTA_DATA}/gold_dev.csv')
df_gold_test = pd.read_csv(f'{RUTA_DATA}/gold_test.csv')
print(f"gold_dev (humano-verificado): {len(df_gold_dev)} comentarios")
print(f"gold_test (humano-verificado, reservado para el Paso 6): {len(df_gold_test)} comentarios")

Pool filtrado (etiquetado por el LLM anotador, confianza >= 0.67): 100858 comentarios
gold_dev (humano-verificado): 128 comentarios
gold_test (humano-verificado, reservado para el Paso 6): 512 comentarios


## 2. Tomar 90.000 del pool y partir 80/20

In [4]:
N_SELECCION = 90_000
df_seleccion, df_sobrante = train_test_split(
    df_pool, train_size=N_SELECCION, stratify=df_pool['label'], random_state=SEED
)
df_entrenamiento, df_validacion_interna = train_test_split(
    df_seleccion, test_size=0.20, stratify=df_seleccion['label'], random_state=SEED
)

print(f"\nDe los {len(df_pool)} del pool, se seleccionan {len(df_seleccion)} (estratificado por emoción).")
print(f"  -> Entrenamiento (80%): {len(df_entrenamiento)} comentarios")
print(f"  -> Validación interna (20%, análoga al 15% del Paso 3/4): {len(df_validacion_interna)} comentarios")
print(f"Sobrante del pool (no entra al fine-tuning): {len(df_sobrante)} comentarios")
print("\nDistribución de emociones en el set de entrenamiento:")
print(df_entrenamiento['label'].value_counts())


De los 100858 del pool, se seleccionan 90000 (estratificado por emoción).
  -> Entrenamiento (80%): 72000 comentarios
  -> Validación interna (20%, análoga al 15% del Paso 3/4): 18000 comentarios
Sobrante del pool (no entra al fine-tuning): 10858 comentarios

Distribución de emociones en el set de entrenamiento:
label
alegria     23697
enojo       22290
burla       11713
neutral      7728
tristeza     3439
sorpresa     2945
miedo         188
Name: count, dtype: int64


## 3. Conjunto de evaluación preliminar

In [5]:
n_humano_total = len(df_gold_dev) + (len(df_gold_test) if INCLUIR_GOLD_TEST_EN_VALIDACION else 0)

if N_VALIDACION_FINAL_OBJETIVO is not None:
    n_desde_pool_deseado = N_VALIDACION_FINAL_OBJETIVO - n_humano_total
    if n_desde_pool_deseado < 0:
        raise ValueError(
            f"N_VALIDACION_FINAL_OBJETIVO ({N_VALIDACION_FINAL_OBJETIVO}) es menor que los "
            f"{n_humano_total} comentarios humanos que ya de por sí entran -- sube el objetivo."
        )
    n_sobrante_disponible = len(df_sobrante)
    if n_desde_pool_deseado > n_sobrante_disponible:
        print(f"\n(Aviso: solo hay {n_sobrante_disponible} comentarios sobrantes del pool, menos "
              f"de los {n_desde_pool_deseado} que harían falta para llegar a "
              f"{N_VALIDACION_FINAL_OBJETIVO} -- se usan los {n_sobrante_disponible} disponibles, "
              f"sin recortar, y la validación final no queda en un número redondo.)")
    else:
        df_sobrante, _ = train_test_split(
            df_sobrante, train_size=n_desde_pool_deseado, stratify=df_sobrante['label'],
            random_state=SEED
        )
        print(f"\nSe recorta el sobrante del pool de {n_sobrante_disponible} a "
              f"{len(df_sobrante)} comentarios (estratificado por emoción, se descarta el resto "
              f"sin usarlo en ningún lado), para que la validación final quede en exactamente "
              f"{N_VALIDACION_FINAL_OBJETIVO} ({len(df_sobrante)} del pool + {n_humano_total} "
              f"humanos, que NO se tocan).")

df_sobrante = df_sobrante.assign(origen='pool_llm')
df_gold_dev = df_gold_dev.assign(origen='gold_dev_humano')
partes_validacion_final = [df_sobrante, df_gold_dev]

if INCLUIR_GOLD_TEST_EN_VALIDACION:
    df_gold_test_marcado = df_gold_test.assign(origen='gold_test_humano')
    partes_validacion_final.append(df_gold_test_marcado)
    print("\n*** OJO: se está incluyendo gold_test en la validación final. Esto significa que "
          "gold_test deja de estar 'intocado' para el Paso 6 -- confirma esto explícitamente "
          "con tu director si no lo has hecho todavía. ***")

df_validacion_final = pd.concat(partes_validacion_final, ignore_index=True)
df_validacion_final = df_validacion_final.sample(frac=1, random_state=SEED).reset_index(drop=True)

n_humano = int((df_validacion_final['origen'] != 'pool_llm').sum())
n_total = len(df_validacion_final)
print(f"\nValidación final combinada: {n_total} comentarios")
print(f"  -> Humano-verificados (gold): {n_humano} ({100 * n_humano / n_total:.1f}%)")
print(f"  -> Etiquetados por el LLM anotador (pool): {n_total - n_humano} "
      f"({100 * (n_total - n_humano) / n_total:.1f}%)")


Se recorta el sobrante del pool de 10858 a 9360 comentarios (estratificado por emoción, se descarta el resto sin usarlo en ningún lado), para que la validación final quede en exactamente 10000 (9360 del pool + 640 humanos, que NO se tocan).

*** OJO: se está incluyendo gold_test en la validación final. Esto significa que gold_test deja de estar 'intocado' para el Paso 6 -- confirma esto explícitamente con tu director si no lo has hecho todavía. ***

Validación final combinada: 10000 comentarios
  -> Humano-verificados (gold): 640 (6.4%)
  -> Etiquetados por el LLM anotador (pool): 9360 (93.6%)


## 4. Guardar los tres archivos

In [6]:
df_entrenamiento.to_csv(f'{RUTA_DATA}/04b_entrenamiento_72k.csv', index=False)
df_validacion_interna.to_csv(f'{RUTA_DATA}/04b_validacion_interna_18k.csv', index=False)
df_validacion_final.to_csv(f'{RUTA_DATA}/04b_validacion_final_combinada.csv', index=False)

print(f"\nGuardado en {RUTA_DATA}:")
print("  04b_entrenamiento_72k.csv          (72.000 -- reemplaza la muestra balanceada de 3.500)")
print("  04b_validacion_interna_18k.csv     (18.000 -- para vigilar el entrenamiento, tipo Trainer eval)")
print(f"  04b_validacion_final_combinada.csv ({n_total} -- reemplaza gold_dev solo, para el classification_report)")
print("\nUsa estos tres archivos en los 9 notebooks/tandas de aquí en adelante, corriendo esto "
      "una sola vez para que todos entrenen y se evalúen sobre exactamente los mismos comentarios.")


Guardado en /content/drive/MyDrive/RUTA/A/data_prep:
  04b_entrenamiento_72k.csv          (72.000 -- reemplaza la muestra balanceada de 3.500)
  04b_validacion_interna_18k.csv     (18.000 -- para vigilar el entrenamiento, tipo Trainer eval)
  04b_validacion_final_combinada.csv (10000 -- reemplaza gold_dev solo, para el classification_report)

Usa estos tres archivos en los 9 notebooks/tandas de aquí en adelante, corriendo esto una sola vez para que todos entrenen y se evalúen sobre exactamente los mismos comentarios.
